In [1]:
import sys
sys.path.append('../../Simulate/')

import os
import random
import numpy as np
import subprocess

from Bio import SeqIO
from tqdm import tqdm
from scipy.stats import bernoulli
from typing import Dict, Union, Tuple
from threading import Lock
from concurrent.futures import ThreadPoolExecutor

from LockedIterator import LockedIterator
from SetMethylation import SetMethylation
from StreamReads import StreamReads
from StreamHTSIM import StreamHTSIM
from UtilityFunctions import get_htsim_path
from ParseGenome import ParseGenome
from BSReadSim import BSReadSim

In [2]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"

ref_fasta = working_path + "data/ref/BSB_test.fa"
outdir = working_path + "outdir"

In [3]:
self = BSReadSim(ref_fasta=ref_fasta, outdir=outdir, 
                 #meth_db_path='/home/wbguo/iproject/BSReadSim/test/outdir/',
                 n_threads=1, num_reads=1000, 
                 overwrite_db=True, verbose =True, shuffle=False, gzip=False)

Initiating genome...
Initiating methylation profile...

[Initiating meth_db] for chr10...
Filling with beta distribution for chr10...
Processed 187408 sites from contig chr10

[Initiating meth_db] for chr11...
Filling with beta distribution for chr11...
Processed 187844 sites from contig chr11

[Initiating meth_db] for chr12...
Filling with beta distribution for chr12...
Processed 184249 sites from contig chr12

[Initiating meth_db] for chr13...
Filling with beta distribution for chr13...
Processed 142075 sites from contig chr13

[Initiating meth_db] for chr14...
Filling with beta distribution for chr14...
Processed 154158 sites from contig chr14

[Initiating meth_db] for chr15...
Filling with beta distribution for chr15...
Processed 2252 sites from contig chr15


../../Simulate/StreamReads.py:39: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir/sim_1.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')
../../Simulate/StreamReads.py:39: UserWarning: Fastq file exists, will overwrite... /home/wbguo/iproject/BSReadSim/test/outdir/sim_2.fastq
  warnings.warn(f'Fastq file exists, will overwrite... {fastq_file}')


# sequential test

In [4]:
for contig_id in self.count_dict.keys():
    sim_cmd  = self.cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]
    read_gen = LockedIterator(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end)) # only output 1 header for -c TODO
    var_contig, sim_data= next(read_gen)                                    # the first element of generator is variants
    self.current_contig = var_contig                                        # update the profiles
    self.pos_map, self.meth_arr, _ = self.meth_db.load_contig(var_contig)   # [pos_map, meth_arr, status]
    self.variant_profile= self.meth_set.set_var_meth(var_contig, sim_data)  # a dict, can be empty
    
    for _, read_pair in read_gen:
        read1_idx   = random.choice([0, 1]) 
        pattern_idx = random.choice([0, 1]) if self.undirectional else read1_idx
        strand_idx  = random.choice([0, 1]) if read_pair[0]['strand']<0 else read_pair[0]['strand']
        read_pair[1-read1_idx]['read2'] = 1
        read_pair[0]['conv'] = pattern_idx
        read_pair[1]['conv'] = pattern_idx
        read_pair[0]['strand'] = strand_idx
        read_pair[1]['strand'] = strand_idx

        # mask the context
        self.mask_context(read_pair[0])
        self.mask_context(read_pair[1])

        # retrive methy profile
        self.retrive_meth_db(read_pair[0])
        self.retrive_meth_db(read_pair[1])

        # set methylation states
        self.set_context_state(read_pair)

        # bisulfite converted
        self.treat_bisulfite(read_pair[0])
        self.treat_bisulfite(read_pair[1])

        # rev complementary
        self.rev_complement(read_pair)

        # introduce seq errors
        self.add_seq_err(read_pair[0])
        self.add_seq_err(read_pair[1])

        # introduce quality scores
        self.add_qual_score(read_pair[0])
        self.add_qual_score(read_pair[1])

        # output
        self.fastq_out.output_reads(read_pair)

self.fastq_out.close()

Simulating whole genome reads:
Reference genome file: /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa
[main] Calculating the total length and effective length of the reference sequences...
[main] Contig chr10 specified, contig length: 423500, effective length: 423500
[main] No VCF input, will generate SNP randomly if mutation rate is nonzero
[htsim] seed = 1677742839
[sim_core] contig 'chr10': simulate 107 reads...
[sim_core] Generated 107 read pairs, with 14 contain SNP, 0 contain INDEL
Simulating whole genome reads:
Reference genome file: /home/wbguo/iproject/BSReadSim/test/data/ref/BSB_test.fa
[main] Calculating the total length and effective length of the reference sequences...
[main] Contig chr11 specified, contig length: 424000, effective length: 424000
[main] No VCF input, will generate SNP randomly if mutation rate is nonzero
[htsim] seed = 1677742840
[sim_core] contig 'chr11': simulate 109 reads...
[sim_core] Generated 109 read pairs, with 10 contain SNP, 3 contain IN

In [5]:
read_pair

[{'read_id': '@chr15:2669:3041:0',
  'pair': 0,
  'read2': 0,
  'conv': 0,
  'strand': 1,
  'flag_pos': 0,
  'flag_mut': 0,
  'flag_indel': 0,
  'start': 2668,
  'end': 2768,
  'cover_pos': 0,
  'n_sub': 0,
  'n_indel': 0,
  'insert_size': 372,
  'inner_dist': 172,
  'cgr': array([0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0], dtype=int8),
  'seq': array([3, 0, 2, 3, 3, 3, 0, 0, 3, 2, 3, 3, 2, 3, 0, 0, 0, 3, 3, 3, 0, 3,
         3, 2, 0, 0, 0, 0, 0, 2, 2, 2, 3, 3, 3, 2, 3, 0, 0, 2, 3, 2, 2, 0,
         2, 0, 3, 3, 3, 3, 0, 0, 2, 1, 2, 0, 0, 3, 3, 3, 0, 1, 2, 3, 3, 0,
         0, 3, 1, 2, 3, 0, 2, 0, 3, 0, 3, 2, 2, 0, 0, 0, 2, 2, 1, 0, 3, 2,
         0, 3, 3, 3, 0, 3, 3, 3, 0, 3, 1, 1], dtype=int8),
  

In [6]:
from pympler import asizeof
print(asizeof.asizeof(read_pair))

12144


# var_profile and generator

In [ ]:
contig_id = 'chr10'
sim_cmd  = self.cmd_part + ['-c', contig_id] + ['-n', str(self.count_dict[contig_id])]

In [ ]:
' '.join(sim_cmd)

In [ ]:
read_gen = LockedIterator(StreamHTSIM(sim_cmd=sim_cmd, pair_end=self.pair_end))
var_contig, sim_data= next(read_gen)
self.current_contig = var_contig
self.pos_map, self.meth_arr, _ = self.meth_db.load_contig(var_contig)
self.variant_profile= self.meth_set.set_var_meth(var_contig, sim_data)

In [ ]:
var_contig

In [ ]:
len(self.pos_map)

In [ ]:
sim_data # 0-based

In [ ]:
self.variant_profile

In [ ]:
for _, read_pair in read_gen:
    if len(read_pair[0]['seq']) != len(read_pair[0]['ctx']):
        break
    if len(read_pair[1]['seq']) != len(read_pair[1]['ctx']):
        break
    if read_pair[0]['n_sub'] !=0:
        break

# check whole process

In [ ]:
read_pair

In [ ]:
print(f"Read1:{''.join(['ACGT'[i] for i in read_pair[0]['seq']])}\nRead2:{''.join(['ACGT'[i] for i in read_pair[1]['seq']])}")

In [ ]:
read1_idx   = random.choice([0, 1]) if read_pair[0]['strand']<0 else read_pair[0]['strand']
pattern_idx = random.choice([0, 1]) if self.undirectional else read1_idx

self.mask_context(read_pair[0], pattern_idx)
self.mask_context(read_pair[1], pattern_idx)

# retrive methy profile
self.retrive_meth_db(read_pair[0])
self.retrive_meth_db(read_pair[1])

# set methylation states
self.set_context_state(read_pair)

# bisulfite converted
self.treat_bisulfite(read_pair[0])
self.treat_bisulfite(read_pair[1])

# rev complementary
self.rev_complement(read_pair)

# introduce seq errors
self.add_seq_err(read_pair[read1_idx], pattern_idx)
self.add_seq_err(read_pair[1-read1_idx],1-pattern_idx)

# introduce quality scores
self.add_qual_score(read_pair[0])
self.add_qual_score(read_pair[1])

self.fastq_out.output_reads(read_pair, read1_idx, pattern_idx)

In [ ]:
str(self.ref_dict['chr10'][131712:131812].seq.reverse_complement())

# mask

In [ ]:
read_pair

In [ ]:
read1_idx   = random.choice([0, 1]) if read_pair[0]['strand']<0 else read_pair[0]['strand']
pattern_idx = random.choice([0, 1]) if self.undirectional else read1_idx

In [ ]:
[read1_idx, pattern_idx]

In [ ]:
self.mask_context(read_pair[0], pattern_idx)
self.mask_context(read_pair[1], pattern_idx)

In [ ]:
read_pair

# retrive

In [ ]:
self.retrive_meth_db(read_pair[0])
self.retrive_meth_db(read_pair[1])

In [ ]:
read_pair[1]

In [ ]:
read_rec = read_pair[1]
read_meth = np.zeros(self.read_len)
site_flag = np.logical_not(read_rec['ctx'].mask)            # unmasked sites

if np.any(site_flag):                                       # contain methylable bases
    arr_idx  = 2
    read_pos = read_rec['start'] + np.arange(self.read_len)

    if read_rec['flag_pos']:                                # covers mutation position
        if self.asm_sim:
            arr_idx  = 4 if read_rec['flag_mut'] else 3

        if read_rec['n_indel']:                             # handle indel first (offset)
            read_pos += read_rec['ofs']
            indel_site= read_rec['cgr'] == 3
            read_meth[indel_site]= self.fetch_meth_val(read_pos[indel_site], arr_idx, 3)

        if read_rec['n_sub']:
            snp_site = site_flag & (read_rec['cgr'] == 1)   # snp methylable site
            read_meth[snp_site]  = self.fetch_meth_val(read_pos[snp_site],  arr_idx, 1)

        match_site = site_flag & (read_rec['cgr'] == 0)     # match methylable site
        read_meth[match_site] = self.fetch_meth_val(read_pos[match_site],arr_idx, 0)
    else:
        match_site = site_flag                              # SNP/INDEL free region
        read_meth[match_site] = self.fetch_meth_val(read_pos[match_site],arr_idx, 0)
read_rec['meth']   = read_meth
read_rec['pos']    = read_pos

In [ ]:
read_pair

In [ ]:
read_pos

In [ ]:
self.variant_profile

# set context state

In [ ]:
self.set_context_state(read_pair)

In [ ]:
read_pair

# treat bisulfite

In [ ]:
self.treat_bisulfite(read_pair[0])
self.treat_bisulfite(read_pair[1])

In [ ]:
read_rec = read_pair[1]

In [ ]:
unmeth_idx = np.where(np.bitwise_and(read_rec['ctx'], 0x1)==1)[0] # behave strange without ==1

In [ ]:
np.where(np.bitwise_and(read_rec['ctx'], 0x1)==1)[0]

In [ ]:
read_rec['meth'].size

In [ ]:
unmeth_idx

In [ ]:
conv_states= bernoulli.rvs(self.conversion_rate, size=unmeth_idx.size)

In [ ]:
conv_states==1

In [ ]:
unmeth_idx

In [ ]:
unmeth_idx[conv_states==1]

In [ ]:
read_pair

In [ ]:
print(f"Read1:{''.join(['ACGT'[i] for i in read_pair[0]['seq']])}\nRead2:{''.join(['ACGT'[i] for i in read_pair[1]['seq']])}")

# reverse complement

In [ ]:
self.rev_complement(read_pair)

In [ ]:
read_pair

In [ ]:
print(f"Read1:{''.join(['ACGT'[i] for i in read_pair[0]['seq']])}\nRead2:{''.join(['ACGT'[i] for i in read_pair[1]['seq']])}")

# add seq err

In [ ]:
self.add_seq_err(read_pair[read1_idx], pattern_idx)
self.add_seq_err(read_pair[1-read1_idx],1-pattern_idx)

In [ ]:
read_rec = read_pair[read1_idx]
pattern_idx = pattern_idx

for i in range(100000):
    err_idx = np.where(bernoulli.rvs(self.err_rate, size = self.read_len))[0] #cannot np.squeeze
    if np.any(err_idx):
        for idx in err_idx:
            base_ori = read_rec['seq'][idx]
            base_err = np.random.choice(np.setdiff1d(np.array([0,1,2,3]), base_ori), size = 1)[0]
            read_rec['seq'][idx] = base_err
            read_rec['cgr'][idx] = 2
            read_rec['ctx'][idx] = 5 if (base_ori, base_err)==[(1,3), (2,0)][pattern_idx] else 6

In [ ]:
read_pair

In [ ]:
err_idx =np.where(bernoulli.rvs(self.err_rate, size = self.read_len))

In [ ]:
err_idx

In [ ]:
read_rec = read_pair[1]
for idx in err_idx:
    base_ori = read_rec['seq'][idx]

In [ ]:
base_ori

In [ ]:
base_err = np.random.choice(np.setdiff1d(np.array([0,1,2,3]), base_ori), size = 1)

In [ ]:
base_err

In [ ]:
print(f"Read1:{''.join(['ACGT'[i] for i in read_pair[0]['seq']])}\nRead2:{''.join(['ACGT'[i] for i in read_pair[1]['seq']])}")

# add qual score

In [ ]:
self.add_qual_score(read_pair[0])
self.add_qual_score(read_pair[1])

In [ ]:
read_pair

# output

In [ ]:
self.fastq_out.output_reads(read_pair, read1_idx, pattern_idx)

In [ ]:
read_rec = read_pair[1]
for ix, ctx in enumerate(read_rec["ctx"]):
    print(ctx)

In [ ]:
read_rec["ctx"]

In [ ]:
read_rec["cgr"]

# save pickle

In [ ]:
# import pickle

# with open('/home/wbguo/iproject/BSReadSim/test/data/test_read_pair.pickle', 'wb') as handle:
#     pickle.dump(read_pair, handle)

# get reference

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq

In [ ]:
str(self.ref_dict['chr10'][18901:19001].seq)

In [ ]:
x = Seq("GAAATACAGATTCCTCGGCACCACCCGAGACCTACTGAATCAGACACAGTAGTAAAAATAAAGATAGTAGGGGCCAGGCGCGGTGGCTCATACCTGTAAC")

In [ ]:
str(x.reverse_complement())